# Silver Validation (Great Expectations)
This notebook runs after `silver_flatten` in the same Databricks job and validates the
`silver_prices` table with a Great Expectations suite. The flatten notebook already routes bad
rows to the dead-letter queue, so this is the second line of defence: it checks that the routing
actually worked and that the table as a whole still honours its contract.

- **Critical** expectations fail the task, which fails the Airflow `run_silver` task.
- **Warning** expectations are recorded but don't stop the run.
- Every result is appended to `silver.dq_results`, so data quality is queryable over time.

## Install
Pinned so a new release can't change behaviour underneath the job. 1.23.2 also includes the fix for
SQLAlchemy 2.1, which broke earlier 1.x releases.

In [ ]:
%pip install great_expectations==1.23.2 --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
import uuid
from datetime import datetime, timezone

import great_expectations as gx
import great_expectations.expectations as gxe
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

spark.sql("USE CATALOG osrs_pipeline")
spark.sql("USE SCHEMA silver")

SILVER_TABLE = "osrs_pipeline.silver.silver_prices"
DIM_ITEMS_TABLE = "osrs_pipeline.silver.dim_items"
RESULTS_TABLE = "osrs_pipeline.silver.dq_results"
SUITE_NAME = "silver_prices_suite"

## The suite
The expectations mirror the Silver contract:

- **Shape:** the table isn't empty and the columns match the declared schema exactly.
- **Grain:** `(item_id, window_timestamp)` is unique, since it's the `MERGE` key.
- **DLQ rules:** volumes are present and non-negative, and any price that is present is positive.
  Nulls in the price columns are allowed because they mean nothing traded on that side.
- **Derived columns:** no row has both prices null, and `is_complete` is true exactly when both
  prices are present.

The warnings are sanity checks I want visibility on without blocking the pipeline: numeric item
IDs, windows on a 5-minute boundary, and every `item_id` existing in `dim_items`. That last one
fails occasionally for a known reason (the OSRS Wiki mapping lags new items), which is why it's a
warning rather than critical.

In [ ]:
CRITICAL, WARNING = "critical", "warning"

SILVER_COLUMNS = [
    "window_timestamp", "item_id", "avg_high_price", "high_price_volume",
    "avg_low_price", "low_price_volume", "is_complete", "window_timestamp_utc",
]


def build_suite():
    suite = gx.ExpectationSuite(name=SUITE_NAME)
    expectations = [
        # Shape
        gxe.ExpectTableRowCountToBeBetween(
            min_value=1, severity=CRITICAL,
            description="Silver is never empty after a run"),
        gxe.ExpectTableColumnsToMatchSet(
            column_set=SILVER_COLUMNS, exact_match=True, severity=CRITICAL,
            description="Schema matches the Silver contract exactly"),
        # Grain
        gxe.ExpectCompoundColumnsToBeUnique(
            column_list=["item_id", "window_timestamp"], severity=CRITICAL,
            description="One row per (item_id, window_timestamp): the MERGE key"),
        gxe.ExpectColumnValuesToNotBeNull(
            column="item_id", severity=CRITICAL, description="item_id is always present"),
        gxe.ExpectColumnValuesToNotBeNull(
            column="window_timestamp", severity=CRITICAL,
            description="window_timestamp is always present"),
        # DLQ rules: these should never fail if the routing in silver_flatten works
        gxe.ExpectColumnValuesToNotBeNull(
            column="high_price_volume", severity=CRITICAL,
            description="high_price_volume is always present"),
        gxe.ExpectColumnValuesToNotBeNull(
            column="low_price_volume", severity=CRITICAL,
            description="low_price_volume is always present"),
        gxe.ExpectColumnValuesToBeBetween(
            column="high_price_volume", min_value=0, severity=CRITICAL,
            description="high_price_volume is never negative"),
        gxe.ExpectColumnValuesToBeBetween(
            column="low_price_volume", min_value=0, severity=CRITICAL,
            description="low_price_volume is never negative"),
        # Column-map expectations skip nulls, so these only check prices that exist
        gxe.ExpectColumnValuesToBeBetween(
            column="avg_high_price", min_value=1, severity=CRITICAL,
            description="avg_high_price is positive when present"),
        gxe.ExpectColumnValuesToBeBetween(
            column="avg_low_price", min_value=1, severity=CRITICAL,
            description="avg_low_price is positive when present"),
        # Derived columns
        gxe.UnexpectedRowsExpectation(
            unexpected_rows_query=(
                "SELECT * FROM {batch} WHERE avg_high_price IS NULL AND avg_low_price IS NULL"
            ),
            severity=CRITICAL,
            description="No rows with both prices null (these are dropped in Silver)"),
        gxe.UnexpectedRowsExpectation(
            unexpected_rows_query=(
                "SELECT * FROM {batch} WHERE is_complete IS NULL OR "
                "is_complete <> (avg_high_price IS NOT NULL AND avg_low_price IS NOT NULL)"
            ),
            severity=CRITICAL,
            description="is_complete is true exactly when both prices are present"),
        # Warnings
        gxe.ExpectColumnValuesToMatchRegex(
            column="item_id", regex=r"^[0-9]+$", severity=WARNING,
            description="Item IDs are numeric strings"),
        gxe.UnexpectedRowsExpectation(
            unexpected_rows_query="SELECT * FROM {batch} WHERE window_timestamp % 300 <> 0",
            severity=WARNING,
            description="Windows sit on a 5-minute boundary"),
        gxe.UnexpectedRowsExpectation(
            unexpected_rows_query=(
                "SELECT b.* FROM {batch} b "
                f"LEFT ANTI JOIN {DIM_ITEMS_TABLE} d ON b.item_id = d.item_id"
            ),
            severity=WARNING,
            description="Every item_id exists in dim_items (mapping can lag new items)"),
    ]
    for expectation in expectations:
        suite.add_expectation(expectation)
    return suite

## Run the validation
An ephemeral context keeps everything in memory, so nothing has to be written to the local
filesystem, which doesn't persist on serverless anyway. The results table below is the durable
record instead.

`persist=False` matters here: by default GX calls `persist()` on the DataFrame before validating,
and serverless compute rejects that.

The whole table is validated each run rather than just the latest window, because the flatten job
re-reads all of Bronze and MERGEs it, so any row can change on any run.

In [ ]:
context = gx.get_context(mode="ephemeral")

source = context.data_sources.add_spark(name="silver", persist=False)
asset = source.add_dataframe_asset(name="silver_prices")
batch_definition = asset.add_batch_definition_whole_dataframe("full_table")

suite = context.suites.add(build_suite())
validation = context.validation_definitions.add(
    gx.ValidationDefinition(name="silver_prices_validation", data=batch_definition, suite=suite)
)

silver_df = spark.table(SILVER_TABLE)
results = validation.run(batch_parameters={"dataframe": silver_df})
print(f"Suite passed: {results.success}")

## Persist the results
One row per expectation per run, appended to `silver.dq_results`. Unlike the Silver tables this is
an append, not a MERGE: each run is its own event with its own `run_id`, so there's nothing to
de-duplicate. Over time this becomes a history of data quality that can be queried or charted.

In [ ]:
RESULTS_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("validated_at", TimestampType(), False),
    StructField("suite_name", StringType(), False),
    StructField("expectation_type", StringType(), False),
    StructField("column_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("severity", StringType(), False),
    StructField("success", BooleanType(), False),
    StructField("element_count", LongType(), True),
    StructField("unexpected_count", LongType(), True),
    StructField("unexpected_percent", DoubleType(), True),
    StructField("observed_value", StringType(), True),
])


def to_rows(validation_results):
    run_id = str(uuid.uuid4())
    validated_at = datetime.now(timezone.utc)
    rows = []
    for r in validation_results.results:
        config = r.expectation_config
        kwargs = config.kwargs
        if kwargs.get("column"):
            column_name = kwargs["column"]
        elif kwargs.get("column_list"):
            column_name = ", ".join(kwargs["column_list"])
        else:
            column_name = None

        result = r.result or {}
        observed = result.get("observed_value")
        unexpected = result.get("unexpected_count")
        if unexpected is None and config.type == "unexpected_rows_expectation":
            # This expectation reports the number of offending rows as its observed value
            unexpected = observed
        pct = result.get("unexpected_percent")
        severity = getattr(config.severity, "value", str(config.severity))

        rows.append((
            run_id,
            validated_at,
            SUITE_NAME,
            config.type,
            column_name,
            config.to_json_dict().get("description"),
            severity,
            bool(r.success),
            result.get("element_count"),
            unexpected,
            None if pct is None else float(pct),
            None if observed is None else str(observed),
        ))
    return rows


results_df = spark.createDataFrame(to_rows(results), RESULTS_SCHEMA)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RESULTS_TABLE} (
    run_id STRING,
    validated_at TIMESTAMP,
    suite_name STRING,
    expectation_type STRING,
    column_name STRING,
    description STRING,
    severity STRING,
    success BOOLEAN,
    element_count BIGINT,
    unexpected_count BIGINT,
    unexpected_percent DOUBLE,
    observed_value STRING
)
USING DELTA
""")

results_df.write.format("delta").mode("append").saveAsTable(RESULTS_TABLE)

display(
    results_df.select("severity", "success", "description", "unexpected_count", "observed_value")
    .orderBy("success", "severity")
)

## Fail on critical expectations
Results are written before this check so a failing run is still recorded. Raising here fails the
task, which fails the job run, which fails `run_silver` in Airflow. Warnings are printed but don't
raise.

In [ ]:
failed = results_df.filter("NOT success").collect()
critical = [row for row in failed if row.severity == CRITICAL]
warnings = [row for row in failed if row.severity == WARNING]

for row in warnings:
    print(f"WARNING: {row.description} (unexpected: {row.unexpected_count})")

if critical:
    summary = "; ".join(f"{row.description} (unexpected: {row.unexpected_count})" for row in critical)
    raise RuntimeError(f"{len(critical)} critical expectation(s) failed: {summary}")

print(f"All critical expectations passed ({len(warnings)} warning(s)).")